# Nyfasi TRELLIS.2 image-to-3D experiment (Google Colab)

Alternative execution route retained alongside the official Hugging Face Space and local CUDA script. Select a Linux NVIDIA GPU runtime. The TRELLIS.2 model requires at least 24 GB VRAM; availability, quota and session duration depend on Colab. This generates a visual hypothesis, not dimensionally accurate CAD.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
p=torch.cuda.get_device_properties(0)
print('GPU:',p.name,'VRAM GiB:',round(p.total_memory/1024**3,1))
assert p.total_memory/1024**3 >= 23.5, 'TRELLIS.2 requires at least 24 GB GPU memory.'
!nvidia-smi

## 1. Download the project and reference photographs
The script records source attribution, dimensions and SHA-256 hashes. Images are research inputs; public availability is not permission to redistribute them.

In [ ]:
BRANCH='feat/nyfasi-reference-images-trellis2'
!git clone --depth 1 --branch $BRANCH https://github.com/andiekobbietks/product-engineering-lab.git /content/product-engineering-lab
%cd /content/product-engineering-lab
!pip -q install Pillow
!python experiments/001-nyfasi-reconstruction/scripts/download_reference_images.py
import json
from pathlib import Path
refdir=Path('experiments/001-nyfasi-reconstruction/output/reference-images')
manifest=json.loads((refdir/'manifest.json').read_text())
for i,s in enumerate(manifest['sources']): print(i,s['id'],s['filename'],s['width_px'],s['height_px'])
image_index=0  # 0 = PDR hero; change to another index to run a different view
selected=manifest['sources'][image_index]
input_image=refdir/selected['filename']
print('Selected:',selected['id'],input_image)

## 2. Install Microsoft's TRELLIS.2
This setup is substantial and may take a while. It follows the upstream installation instructions; environment compatibility can change as upstream dependencies evolve.

## 4. Inspect the generated mesh with the Python toolchain
This is post-processing only. It does not alter the source GLB or assert physical units.

In [ ]:
%cd /content/product-engineering-lab
!pip -q install -r requirements-reconstruction.txt
!python experiments/001-nyfasi-reconstruction/scripts/process_trellis2_mesh.py /content/nyfasi_trellis2_hypothesis.glb --report /content/nyfasi_mesh_report.json
from google.colab import files
files.download('/content/nyfasi_mesh_report.json')


In [ ]:
%cd /content
!git clone --recursive https://github.com/microsoft/TRELLIS.2.git
%cd /content/TRELLIS.2
!bash -lc 'source ./setup.sh --new-env --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm'

## 3. Run image-to-3D inference
The repository script exports a textured GLB. Inspect the output visually; do not interpret its scale or hidden features as measurements.

In [ ]:
%cd /content/product-engineering-lab
!cp "$input_image" /content/nyfasi_input.png
!cp experiments/001-nyfasi-reconstruction/scripts/run_trellis2_image_to_3d.py /content/TRELLIS.2/run_nyfasi_trellis2.py
%cd /content/TRELLIS.2
!bash -lc 'source ~/miniforge3/etc/profile.d/conda.sh 2>/dev/null || true; conda activate trellis2 && python run_nyfasi_trellis2.py /content/nyfasi_input.png --output /content/nyfasi_trellis2_hypothesis.glb'
from pathlib import Path
glb=Path('/content/nyfasi_trellis2_hypothesis.glb')
assert glb.exists() and glb.stat().st_size>100_000
print('GLB MiB:',round(glb.stat().st_size/1024**2,2))

## 4. Export GLB and provenance
Keep the output separate from Sand3D's canonical assembly. Repeat with open and closed product photographs as separate runs.

In [ ]:
import hashlib, json
from google.colab import files
provenance={'model':'microsoft/TRELLIS.2-4B','source_id':selected['id'],'source_url':selected['url'],'source_sha256':selected['sha256'],'execution_route':'Google Colab','interpretation':'AI-generated visual hypothesis; not dimensionally calibrated CAD'}
Path('/content/nyfasi_trellis2_provenance.json').write_text(json.dumps(provenance,indent=2)+'\\n')
files.download(str(glb))
files.download('/content/nyfasi_trellis2_provenance.json')